# Project Setup

Import Required Libraries

In [ ]:
# Import libraries

# For Colab
import gdown
from google.colab import drive

# For data analysis
import numpy as np
import pandas as pd
import polars as pl
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from yellowbrick.cluster.elbow import kelbow_visualizer
%matplotlib inline

# For garbage collection
import gc

# Regression
from xgboost import XGBRegressor
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import KFold

# Model evaluation tools
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import shap

Load raw data for demographics and household spending

In [ ]:
# Mount Google Drive
drive.mount('/content/drive/')

In [ ]:
# Confirm files are in the expected path
!ls "/content/drive/My Drive/DS3000/coursework_data"

In [ ]:
# Original file locations for A.B.
demo_data = pl.read_csv("/content/drive/My Drive/DS3000/coursework_data/DemoStats.csv", null_values = "NA").sample(fraction = 0.1, seed = 2025)
household_data = pl.read_csv("/content/drive/My Drive/DS3000/coursework_data/HouseholdSpend.csv", null_values = "NA").sample(fraction = 0.1, seed = 2025)

# For local work
# demo_data = pl.read_csv("D:\\ds_3000_project\\DemoStats.csv", null_values = "NA").sample(fraction = 0.1, seed = 2025)
# household_data = pl.read_csv("D:\\ds_3000_project\\HouseholdSpend.csv", null_values = "NA").sample(fraction = 0.1, seed = 2025)

Preview Initial Dataset Dimensions

In [ ]:
print(f"demo_data shape: {demo_data.shape}")
print(f"household_data shape: {household_data.shape}")

# Data Cleaning & Preparation

Drop zero-only rows (Household)

In [ ]:
###
#
# Cleaning out all zero only rows
# DemoStats data has no rows that have 0s
#
###

# Convert Househould Data into pandas since we are checking by rows
household_data = household_data.to_pandas()

# Select only numeric columns
household_numeric_data = household_data.select_dtypes(include='number')

# Create a boolean mask for rows where all numeric columns equal 0
house_mask = (household_numeric_data == 0).all(axis=1)

# Count the rows that are only zeros
house_zero_row_count = house_mask.sum()
print("Household Rows with only zeros:", house_zero_row_count)

# Remove the rows: use mask.values (a 1D numpy array) for indexing
household_data = household_data.loc[house_mask.values == False]

# Optional: reset the index
household_data = household_data.reset_index(drop=True)

# Convert Household Data back to polars
household_data = pl.from_pandas(household_data)

# Deleting unnecessary variables
# del [household_numeric_data, house_mask, house_zero_row_count]
del household_numeric_data, house_mask, house_zero_row_count

# Trigger Garbage Collection
gc.collect()

Make sure 0-filled rows were removed (Household)

In [ ]:
# Convert again to pandas for checking
household_data_check = household_data.to_pandas()

# Select numeric columns
numeric_check = household_data_check.select_dtypes(include='number')

# Check if any rows are still entirely zero
remaining_zeros = (numeric_check == 0).all(axis=1).sum()
print("Remaining zero-only rows after cleanup:", remaining_zeros)

# Cleanup variables
del household_data_check, numeric_check, remaining_zeros
gc.collect()

Drop zero-only rows (Demographics)

In [ ]:
###
#
# Cleaning out all zero only rows (Demographics dataset)
#
###

# Convert Househould Data into pandas since we are checking by rows
demo_data = demo_data.to_pandas()

# Select only numeric columns
demo_numeric_data = demo_data.select_dtypes(include='number')

# Create a boolean mask for rows where all numeric columns equal 0
demo_mask = (demo_numeric_data == 0).all(axis=1)

# Count the rows that are only zeros
demo_zero_row_count = demo_mask.sum()
print("Demographic Rows with only zeros:", demo_zero_row_count)

# Remove the rows: use mask.values (a 1D numpy array) for indexing
demo_data = demo_data.loc[demo_mask.values == False]

# Optional: reset the index
demo_data = demo_data.reset_index(drop=True)

# Convert Household Data back to polars
demo_data = pl.from_pandas(demo_data)

# Deleting unnecessary variables
# del [household_numeric_data, house_mask, house_zero_row_count]
del demo_numeric_data, demo_mask, demo_zero_row_count

# Trigger Garbage Collection
gc.collect()

In [ ]:
# Convert again to pandas for checking
demo_data_check = demo_data.to_pandas()

# Select numeric columns
numeric_check = demo_data_check.select_dtypes(include='number')

# Check if any rows are still entirely zero
remaining_zeros = (numeric_check == 0).all(axis=1).sum()
print("Remaining zero-only rows after cleanup:", remaining_zeros)

# Cleanup variables
del demo_data_check, numeric_check, remaining_zeros
gc.collect()

Checking dataset dimensions again

In [ ]:
household_data.shape

In [ ]:
# Note that demographics data has more columns than household data
# # I don't think an observation is useful if demographics data was present but household data was 0-filled
demo_data.shape

Look at the first few rows of the household data

In [ ]:
household_data.head()

Look at the first few rows of the demographics data

In [ ]:
demo_data.head()

Summarizing variable types for the household and demogaphics datasets

In [ ]:
# Summarizing the variable data types for the household dataset

# Create a dataframe with variable names and data types
column_info_df = pl.DataFrame(
    {
        "column": list(household_data.schema.keys()), # Get column names
        "data_type": [str(dtype) for dtype in household_data.schema.values()], # Get and convert data types to string
    }
)

# Get value counts for data types
column_info_df.group_by("data_type").agg(pl.len().alias("count")).sort("count", descending = True)

In [ ]:
# Summarizing the variable data types for the demographic dataset

# Create a dataframe with variable names and data types
column_info_df = pl.DataFrame(
    {
        "column": list(demo_data.schema.keys()), # Get column names
        "data_type": [str(dtype) for dtype in demo_data.schema.values()], # Get and convert data types to string
    }
)

# Get value counts for data types
column_info_df.group_by("data_type").agg(pl.len().alias("count")).sort("count", descending = True)

Inspecting the string columns

In [ ]:
# Checking out the string variables for the household dataset
# Should drop "GEO" because it's not unique to each observation
household_data.select(pl.col(pl.String)).head(5)

In [ ]:
# Checking out the string variables for the demographics dataset
# Should drop "GEO" because it's not unique to each observation
demo_data.select(pl.col(pl.String)).head(5)

Drop Uninformative String Columns

In [ ]:
# Drop the GEO columns from both datasets because they're not unique to each observation
household_data = household_data.drop("GEO")
demo_data = demo_data.drop("GEO")

Remove Zero-Filled Columns (Household)

In [ ]:
# Finding and removing 0-filled columns in the household dataset

# List to store the names of zero-only columns
zero_cols = []

# Loop over all columns, checking each if only zeros are present
# If so, append the column name to the zero_cols list
for col in household_data.columns:

    # Check if all values in this column are zero
    if (household_data[col] == 0).all():
        zero_cols.append(col)

In [ ]:
# No zero-filled columns found in the household dataset
len(zero_cols)

Remove Zero-Filled Columns (Demographic)

In [ ]:
# Finding and removing 0-filled columns in the demographics dataset

# Create an empty list to hold numeric column names
numeric_cols = []

# Identify numeric columns
for col, dtype in demo_data.schema.items():
    if dtype in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64
    ):
        numeric_cols.append(col)

# Create an empty list for zero-filled columns
zero_cols = []

# Check each numeric column for all-zero values (ignoring nulls by treating them as zero)
for col in numeric_cols:
    col_is_not_zero = demo_data.select(
        (pl.col(col).fill_null(0) != 0).any()
    ).item()

    if not col_is_not_zero:
        zero_cols.append(col)

print("Zero-filled columns:", zero_cols)


In [ ]:
# Two zero-filled columns found in the demographics dataset
len(zero_cols)

In [ ]:
# Descriptive statistics for the zero-filled columns in the demographics dataset
# There are only 2: ECYASQKM and ECYALSQKM
# These correspond to "Total Area" and "Total Land Area", respectively
# Should probably drop them
demo_data[zero_cols].describe()

In [ ]:
# Drop the zero-filled columns
demo_data = demo_data.drop(zero_cols)

Remove Zero-Filled Columns (Demographic)

In [ ]:
# Finding and removing 0-filled columns in the household dataset

# Create an empty list to hold numeric column names
numeric_cols = []

# Identify numeric columns
for col, dtype in household_data.schema.items():
    if dtype in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64
    ):
        numeric_cols.append(col)

# Create an empty list for zero-filled columns
zero_cols = []

# Check each numeric column for all-zero values (ignoring nulls by treating them as zero)
for col in numeric_cols:
    col_is_not_zero = household_data.select(
        (pl.col(col).fill_null(0) != 0).any()
    ).item()

    if not col_is_not_zero:
        zero_cols.append(col)

print("Zero-filled columns:", zero_cols)


Handle Negative Values:

For the household dataset

In [ ]:
# Find the columns that contain negative values
has_negatives = []

for index, col in enumerate(household_data.columns):
  if (household_data.dtypes[index] == pl.Int64 or household_data.dtypes[index] == pl.Float64):
    minValue = household_data[col].min()
    if (minValue < 0):
      has_negatives.append(col)

print(f"The number of columns that have negative values is: {len(has_negatives)}.\n\nThese columns are:")

for entry in has_negatives:
  print(f"{entry}")

In [ ]:
# Inspecting the columns that contain negative values, identified above

# Subset the dataframe to only columns with negative values
negatives_df = household_data.select(has_negatives)

# Total number of observations (rows) in the dataset
total_rows = household_data.height

# Count negative values and compute proportion for each column
for col in has_negatives:
    count = negatives_df.filter(negatives_df[col] < 0).height
    proportion = count / total_rows
    print(f"Column '{col}' has {count} negative values ({proportion:.2%} of all observations).")

# HSTT001: Total expenditure
# HSTE001ZBS: Total non-current consumption
# HSWH040S: Net purchase price of owned residences
# HSWH041S: Net purchase price of owned secondary residences
# HSWH042S: Net purchase price of other owned properties

In [ ]:
# Descriptive stats for columns with negative values
household_data[has_negatives].describe()

For the demographics dataset (no columns containing negative values found):

In [ ]:
# Find the columns that contain negative values for the demographics dataset
# None were found
has_negatives = []

for index, col in enumerate(demo_data.columns):
  if (demo_data.dtypes[index] == pl.Int64 or demo_data.dtypes[index] == pl.Float64):
    minValue = demo_data[col].min()
    if (minValue < 0):
      has_negatives.append(col)

print(f"The number of columns that have negative values is: {len(has_negatives)}.\n\nThese columns are:")

for entry in has_negatives:
  print(f"{entry}")

Handle Missing Values (Household data)

In [ ]:
# Checking out missing values in the household dataset

# Make a df with var names in one column, and null counts in the next column
missing_values_df = (
    household_data.select(
        [pl.col(col).is_null().sum().alias(col) for col in household_data.columns] # Count nulls per column
    )
    .transpose(include_header = True) # Convert column names to a single column
    .rename({"column_0": "missing_count"}) # Rename output
    .with_columns([
        pl.col("missing_count").cast(pl.Int64), # Ensure numeric type
        (pl.col("missing_count") / household_data.height).alias("proportion_of_obs_missing")
        ])
    .sort("missing_count", descending = True) # Sort by missing count
)

# Show only variables with missing values
missing_values_df.filter(pl.col("missing_count") > 0)

# Seems there are no missing values in the household dataset now


Handle Missing Values (Demographics data)

In [ ]:
# Checking out missing values in the demographics dataset

# Make a df with var names in one column, and null counts in the next column
missing_values_df = (
    demo_data.select(
        [pl.col(col).is_null().sum().alias(col) for col in demo_data.columns] # Count nulls per column
    )
    .transpose(include_header = True) # Convert column names to a single column
    .rename({"column_0": "missing_count"}) # Rename output
    .with_columns([
        pl.col("missing_count").cast(pl.Int64), # Ensure numeric type
        (pl.col("missing_count") / demo_data.height).alias("proportion_of_obs_missing")
        ])
    .sort("missing_count", descending = True) # Sort by missing count
)

# Show only variables with missing values
missing_values_df.filter(pl.col("missing_count") > 0)

# Found 7 columns with missing values, all have 10% or greater missing


In [ ]:
demo_data.shape

In [ ]:
# Make a list of columns to drop.
# We should discuss whether to impute these values instead.

# However if I remember correctly we should only impute if missing values
# make up less than 1-2% of the column? This is at least 10% missing.

cols_to_drop = (
    missing_values_df
    .filter(pl.col("missing_count") > 0) # Keep only vars with missing values
    .select("column") # Select the column that contains column name strings
    .to_series() # Convert to Polars Series
    .to_list() # Convert to list
)

In [ ]:
# Show cols to drop
demo_data[cols_to_drop].describe()

In [ ]:
# Drop the columns with missing values
demo_data = demo_data.drop(cols_to_drop)

# Merge Datasets
- Merge cleaned demographic and household data on the CODE identifier.

In [ ]:
# Merging
merged_data = household_data.join(demo_data, on = "CODE", how = "left")

# Delete unneeded objects
del demo_data, household_data

# Trigger garbage collection
gc.collect()

In [ ]:
# Note that the number of rows is the same as the household data
merged_data.shape

# Train/Test Split

In [ ]:
# Split merged_data_clustering into training, validation, and testing sets

# Set the seed and fractions for reproducibility
split_seed = 2025
train_frac = 0.7
val_frac = 0.15

# Randomly shuffle the rows with seed
shuffled = merged_data.sample(fraction=1.0, with_replacement=False, seed=split_seed)

# Calculate the indices for where the splits happen
total_rows = shuffled.height
train_index = int(total_rows * train_frac)
val_index = train_index + int(total_rows * val_frac)

# Slice the shuffled dataframe into train, validation, and test sets
train_df = shuffled[:train_index]
val_df = shuffled[train_index:val_index]
test_df = shuffled[val_index:]

# Show subset dimensions
print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Testing set shape: {test_df.shape}")


Check for constant columns

In [ ]:
# Columns with standard deviation of 0 (only numeric columns)
zero_std_columns = [
    col for col in train_df.columns
    if train_df.schema[col] in (pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                          pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                          pl.Float32, pl.Float64)
    and train_df.select(pl.std(col)).item() == 0
]

# Columns that are of type string
string_columns = [
    col for col, dtype in train_df.schema.items()
    if dtype == pl.Utf8
]

print("Columns with standard deviation of 0:", zero_std_columns)
print("Columns of type string:", string_columns)

# Outlier Handling

## Define and Apply Z-Score Outlier Cleaning on Train Set
- Replace low-percentage outliers with medians.

In [ ]:
# Modified version of Joel's outlier cleaning function
# Also returns dicts that contain the columns medians, and a dict with columns means and stdevs that are required for processing the test dataframe

def z_score_cleaning(df, numeric_cols, THRESHOLD_PERCENTAGE):
    columns_with_many_outliers = {}
    column_medians = {}
    column_stats = {}

    print("Initializing Cleaning")
    for col in numeric_cols:
        stats = df.select([
            pl.col(col).mean().alias("mean"),
            pl.col(col).std().alias("std"),
            pl.col(col).median().alias("median")
        ]).row(0)

        col_mean, col_std, col_median = stats
        column_medians[col] = col_median
        column_stats[col] = (col_mean, col_std)

        lower_bound = col_mean - 2.5 * col_std
        upper_bound = col_mean + 2.5 * col_std

        outlier_mask = (pl.col(col) < lower_bound) | (pl.col(col) > upper_bound)

        outlier_count = df.filter(outlier_mask).height
        outlier_percentage = (outlier_count / df.height) * 100

        if outlier_percentage < THRESHOLD_PERCENTAGE:
            df = df.with_columns([
                pl.when(outlier_mask)
                  .then(col_median)
                  .otherwise(pl.col(col))
                  .alias(col)
            ])
        else:
            columns_with_many_outliers[col] = df.filter(outlier_mask)

    print("Cleaning has concluded")
    return df, columns_with_many_outliers, column_medians, column_stats

In [ ]:
# Function that applies the medians calculated above (on the training data) to the test data outliers

def apply_medians_to_test(df_test, column_medians, column_stats):
    """
    Applies median replacement to the test dataframe using training data medians and stats.

    Arguments:
        df_test (pl.DataFrame): The test DataFrame.
        column_medians (dict): Dictionary of medians from training data.
        column_stats (dict): Dictionary of (mean, std) tuples from training data.

    Returns:
        pl.DataFrame: Cleaned test DataFrame.
    """
    for col in column_medians:
        col_median = column_medians[col]
        col_mean, col_std = column_stats[col]

        lower_bound = col_mean - 2.5 * col_std
        upper_bound = col_mean + 2.5 * col_std

        outlier_mask = (pl.col(col) < lower_bound) | (pl.col(col) > upper_bound)

        # Replace outliers with the training median
        df_test = df_test.with_columns([
            pl.when(outlier_mask)
              .then(col_median)
              .otherwise(pl.col(col))
              .alias(col)
        ])

    return df_test


In [ ]:
train_df_numeric_cols = [col for col, dtype in train_df.schema.items()
                if dtype in (pl.Int64, pl.Float64, pl.Int32, pl.Float32)]

## Clean Test Set Using Train Medians

In [ ]:
THRESHOLD_PERCENTAGE = 5.0

# Clean training data and extract medians + stats
df_train_cleaned, outliers_dict, medians_dict, stats_dict = z_score_cleaning(train_df, train_df_numeric_cols, THRESHOLD_PERCENTAGE)

Check for constant columns

In [ ]:
# Columns with standard deviation of 0 (only numeric columns)
zero_std_columns = [
    col for col in df_train_cleaned.columns
    if df_train_cleaned.schema[col] in (pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                          pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                          pl.Float32, pl.Float64)
    and df_train_cleaned.select(pl.std(col)).item() == 0
]

# Columns that are of type string
string_columns = [
    col for col, dtype in df_train_cleaned.schema.items()
    if dtype == pl.Utf8
]

print("Columns with standard deviation of 0:", zero_std_columns)
print("Columns of type string:", string_columns)

In [ ]:
# Clean test data using saved training medians and stats
df_test_cleaned = apply_medians_to_test(test_df, medians_dict, stats_dict)

In [ ]:
# Clean validation data using saved training medians and stats
df_val_cleaned = apply_medians_to_test(val_df, medians_dict, stats_dict)

In [ ]:
# Columns with standard deviation of 0 (only numeric columns)
zero_std_columns = [
    col for col in df_test_cleaned.columns
    if df_test_cleaned.schema[col] in (pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                          pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                          pl.Float32, pl.Float64)
    and df_test_cleaned.select(pl.std(col)).item() == 0
]

# Columns that are of type string
string_columns = [
    col for col, dtype in df_test_cleaned.schema.items()
    if dtype == pl.Utf8
]

print("Columns with standard deviation of 0:", zero_std_columns)
print("Columns of type string:", string_columns)

In [ ]:
len(zero_std_columns)

In [ ]:
df_test_cleaned[zero_std_columns]

# Clustering – Part 1 of Assignment

## Drop Columns Correlated with Target
- Exclude variables later used for the regression target variable.

In [ ]:
# Dropping the columns that will be used to create the target variable later

# Variables used for calculating target variable:
# Household Income: HSHNIAGG
# Total personal insurance premiums and retirement/pension contributions: HSEP001S
# Household Disposable Income: HSAGDISPIN
# Household Discretionary Income: HSAGDISCIN
# Income tax: HSTX001 (kinda 50/50 on if we should drop this one)

# Are there other columns we need to drop?


# Drop the variables used for calculating target variable
# (and also drop variables correlated to the target)
# df_train_cleaned_clustering = df_train_cleaned.drop(["HSHNIAGG", "HSAGDISPIN", "HSAGDISCIN", "HSEP001S"])
df_train_cleaned_clustering = df_train_cleaned.drop(["HSHNIAGG", "HSEP001S"])

In [ ]:
df_train_cleaned_clustering.shape

In [ ]:
# Identify and count all columns that start with "HSEP"
drop_HSEP = [col for col in df_train_cleaned_clustering.columns if col.startswith("HSEP")]
len(drop_HSEP)

## Subset and Standardize for Clustering
- Randomly sample and standardize numeric features.

In [ ]:
# Subset the Polars df to just the numeric variables by dropping the "CODE" col
# and store as a numpy array.

# Currently further downsampling to 50% for clustering

# Also note there is a seed to set, or we could just let it be random.
X = df_train_cleaned_clustering.drop("CODE").sample(fraction = 0.5, with_replacement = False, seed = 2025).to_numpy()

In [ ]:
# Standardize the data.
# This means every numeric column should have a mean of 0 and sd of 1.

# Initialize the scaler
scaler = StandardScaler()

# Use the scaler to normalize X
X = scaler.fit_transform(X)

In [ ]:
# Just confirming that the scaling worked correctly.
# Compute column-wise means and standard deviations
means = np.mean(X, axis = 0)
stdevs = np.std(X, axis = 0)

In [ ]:
# Show the column-wise means.
# If the scaling step worked correctly, these should all be very close to zero.
means

In [ ]:
# Print the column-wise standard deviations.
# If the scaling step worked correctly, these should all be 1.
stdevs

## KMeans Clustering with Elbow Method
- Determine optimal number of clusters using visual elbow method.

In [ ]:
# Initializing the cluster algorithm
KClusterer = KMeans(random_state = 2025)

# Initialize the elbow visualizer.
# This code was basically copied straight from Mar.20 lab.
visualizer = KElbowVisualizer(KClusterer, # Cluster model with any parameters you need
                              k = (2, 12), # Number of clusters to test (2 to 12 in this case)
                              locate_elbow = True, # Locate the elbow? Default is true.
                              timings = False # Plot the timings to train?
                             )

visualizer.fit(X) # Fit the visualizer
visualizer.show()

## Silhouette Scores

Two Approaches below:
1. Calculates Silhouette score and also provides Silhouette graph
2. Calculates Average Silhouette scores and presents them without a graph. There is a final graph that shows all scores at the end of run.

*Currently Commented out since it takes a long time to run*

In [ ]:
# Silhouette with graphs
# Takes too long to run

# Loop through cluster counts from 2 to 9.
for n_clusters in range(2, 10):
    print("="*50)
    print(f"Evaluating KMeans with {n_clusters} clusters")

    # Create a pipeline: scale data and then apply KMeans.
    kmeans_pipe = Pipeline([
        # ('scale', StandardScaler()), # X is already scaled in a cell above
        ('kmeans', KMeans(n_clusters=n_clusters, n_init=10, random_state=2025))
    ])

    # Fit the pipeline to the training data.
    kmeans_pipe.fit(X)

    # Predict the cluster labels.
    cluster_labels = kmeans_pipe.predict(X)

    # Transform X_train using the scaler from the pipeline.
    # X_train_scaled = kmeans_pipe.named_steps['scale'].transform(X_train_scaled)

    # Calculate the average silhouette score.
    sil_score = silhouette_score(X, cluster_labels)
    print(f"Average silhouette score for k = {n_clusters}: {sil_score:.3f}")

    # Initialize and fit the Yellowbrick SilhouetteVisualizer.
    visualizer = SilhouetteVisualizer(kmeans_pipe.named_steps['kmeans'], colors='yellowbrick')
    visualizer.fit(X)
    visualizer.show()

In [ ]:
# Only Silhouette Averages are calculated here
# has a graph to show at the end (Not silhouette graph)
# Takes too long to run

silhouette_avgs = []

# Loop through cluster counts from 2 to 13.
for n_clusters in range(2, 9):
    # Initialize and fit the KMeans model
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=2025)
    cluster_labels = kmeans.fit_predict(X)

    # Calculate the average silhouette score for the current number of clusters
    silhouette_avg = silhouette_score(X, cluster_labels)
    silhouette_avgs.append(silhouette_avg)
    print(f"For n_clusters = {n_clusters}, the average silhouette score is {silhouette_avg:.4f}")

## Fit Final KMeans and Assign Labels

In [ ]:
# Now that we've determined the optimal number of clusters, fit the final K-means

# Initialize and fit K-means
KClusterer = KMeans(n_clusters = 2, random_state = 2025)
KClusterer.fit(X)

# Get cluster labels for plotting later on
# This tells matplotlib or seaborn what colour each observation should be, based on cluster membership
labels = KClusterer.labels_

## Dimensionality Reduction with PCA

In [ ]:
# Principal component analysis

# Initialize PCA with 5 PCs.
pca = PCA(n_components = 5)

# Fit PCA to the standardized data and transform
X_pca = pca.fit_transform(X)

In [ ]:
# Look at the proportion of explain variance associated with each PC
# PC1 explains most of the variance
print(pca.explained_variance_ratio_)

In [ ]:
# Create a new Seaborn scatter plot
sns.scatterplot(
    x = X_pca[:, 0], # PC1
    y = X_pca[:, 1], # PC2
    hue = labels, # Base the number of unique legend colours on the labels we saved previously
    palette = 'tab10',
    alpha = 0.5,
    edgecolor = 'k'
)

plt.title("PCA of Regional Districts")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title = 'Cluster', loc = 'best')

## Interpret PCA Loadings

In [ ]:
# Create a dataframe from PCA output using the first 3 components
# If changing the number of components, change the slicing of X_pca accordingly
df_pca = pd.DataFrame(X_pca[:, 0:3], columns = ["PC1", "PC2", "PC3"])

# Add cluster labels (which were stored previously)
df_pca["cluster"] = labels

# Group by cluster and compute average of each PC
# Apparently we don't need .agg() here because .mean() already does that
cluster_avg = df_pca.groupby("cluster").mean()

# Display the result
print(cluster_avg)

# Based on the PC loadings analysis below:
# Cluster 0: Slightly larger than average population, some cultural diversity, average incomes
# Cluster 1: Very large population but low-income and low-diversity
# Cluster 2: Large multicultural neighbourhoods with decent incomes
# Cluster 3: Slightly smaller than average population, average diversity and incomes

In [ ]:
# Get colnames from original dataframe without "CODE" column
feature_names = df_train_cleaned_clustering.drop("CODE").columns

# Create a dataframe of PCA loadings
loadings = pd.DataFrame(pca.components_,
                        columns = feature_names, # Set variable names as column labels
                        index = [f"PC{i+1}" for i in range(pca.n_components_)]) # Name rows as PC1, PC2, etc.

print(loadings)


In [ ]:
# Making a dataframe of variable names and their descriptions
# This will be used for matching variable descriptions to their codes
# when inspecting PC loadings.
demo_metadata = pd.read_csv("/content/drive/My Drive/DS3000/coursework_data/DemoStats 2024 - Metadata.csv")
# demo_metadata = pd.read_csv("D:\\ds_3000_project\\DemoStats 2024 - Metadata.csv") # For local work
demo_metadata = demo_metadata[["Variable", "Description"]]

household_metadata = pd.read_csv("/content/drive/My Drive/DS3000/coursework_data/HouseholdSpend 2024 - Metadata.csv")
# household_metadata = pd.read_csv("D:\\ds_3000_project\\coursework_data/HouseholdSpend 2024 - Metadata.csv")
household_metadata = household_metadata[["Variable", "Description"]]
household_metadata = household_metadata.drop([0, 1]) # Drop the first two rows because they already exist in the demographics metadata
household_metadata = household_metadata.reset_index(drop = True) # Reset the index

metadata_df = pd.concat([demo_metadata, household_metadata], ignore_index = True)

In [ ]:
# Inspecting the loadings for PC1
(loadings.loc["PC1"]
    .sort_values(ascending = False)
    .reset_index().rename(columns = {"index" : "Variable"})
    .merge(metadata_df, on = "Variable", how = "left")
    .head(20))

# Based on the loadings, we'd call PC1 "Neighbourhoods with large populations"

In [ ]:
# Inspecting the loadings for PC2
(loadings.loc["PC2"]
    .sort_values(ascending = False)
    .reset_index().rename(columns = {"index" : "Variable"})
    .merge(metadata_df, on = "Variable", how = "left")
    .head(20))

# Based on the loadings, we'd call PC2 "High Presence of Immigrant Households"

In [ ]:
# Inspecting the loadings for PC3
(loadings.loc["PC3"]
    .sort_values(ascending = False)
    .reset_index().rename(columns = {"index" : "Variable"})
    .merge(metadata_df, on = "Variable", how = "left")
    .head(20))

# Based on the loadings, we'd call PC2 "Wealthy Neighbourhoods"

## Alternative Visualization Using UMAP

In [ ]:
import umap

# Store best results
best_silhouette = -1
best_embedding = None
best_params = {}

# Try combinations of neighbors and min_dist
neighbors_options = [10, 30, 50]
min_dist_options = [0.0, 0.3, 0.8]

for n in neighbors_options:
    for d in min_dist_options:
        print(f"Training UMAP with n_neighbors={n}, min_dist={d}...")

        # Create UMAP object
        reducer = umap.UMAP(n_neighbors=n,
                            n_components=2,
                            metric='cosine',
                            n_epochs=1000,
                            min_dist=d,
                            spread=1.0,
                            low_memory=False,
                            n_jobs=-1,
                            verbose=True,
                            random_state=2025)

        # Train UMAP and get embedding
        embedding = reducer.fit_transform(X)

        # Evaluate using silhouette score
        score = silhouette_score(embedding, labels)
        print(f"Silhouette Score: {score:.4f}")

        # Save if it's the best so far
        if score > best_silhouette:
            best_silhouette = score
            best_embedding = embedding
            best_params = {'n_neighbors': n, 'min_dist': d}

# Output best result
print("\nBest UMAP Parameters:")
print(best_params)
print(f"Best Silhouette Score: {best_silhouette:.4f}")

In [ ]:
# Plot final UMAP embedding
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x = best_embedding[:, 0],
    y = best_embedding[:, 1],
    hue = labels,
    palette = 'tab10',
    alpha = 0.5,
    edgecolor = 'k'
)

plt.title(f"UMAP Projection (n_neighbors={best_params['n_neighbors']}, min_dist={best_params['min_dist']})")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend(title='Cluster')
plt.grid(True)
plt.tight_layout()
plt.show()

# Regression Modeling – Part 2 of Assignment
- Predict insurance & pension contributions as a proportion of income.

## Train/Validation Split (commented out because I made the validation set earlier now)

## Create Target Variable

In [ ]:
target_col_name = "insurance_pension_ratio"
col_contrib = "HSEP001S"
col_income = "HSHNIAGG"

# Function to calculate the ratio
def calculate_target(df, contrib_col, income_col, target_name):
    return df.with_columns(
        (pl.col(contrib_col) / pl.col(income_col)).alias(target_name)
    )

# Calculate targets for train, val, and test
df_train_final_xgb = calculate_target(train_df.drop("CODE"), col_contrib, col_income, target_col_name)
df_val_xgb = calculate_target(val_df.drop("CODE"), col_contrib, col_income, target_col_name)
df_test_xgb = calculate_target(test_df.drop("CODE"), col_contrib, col_income, target_col_name)

In [ ]:
# Checking target variable descriptive stats
print("Target Variable Descriptive Statistics (Training Set):")
print(df_train_final_xgb.select(target_col_name).describe())

print("Target Variable Descriptive Statistics (Validation Set):")
print(df_val_xgb.select(target_col_name).describe())

print("\nTarget Variable Descriptive Statistics (Test Set):")
print(df_test_xgb.select(target_col_name).describe())

## Extract Final Features and Targets for Modeling

In [ ]:
# Drop unnecessary columns
cols_to_drop_for_xgb = [target_col_name, col_contrib, col_income]

# Extract features and target for training and validation
X_train_pl = df_train_final_xgb.drop(cols_to_drop_for_xgb)
y_train_pl = df_train_final_xgb.select(target_col_name)

X_val_pl = df_val_xgb.drop(cols_to_drop_for_xgb)
y_val_pl = df_val_xgb.select(target_col_name)

X_test_pl = df_test_xgb.drop(cols_to_drop_for_xgb)
y_test_pl = df_test_xgb.select(target_col_name)

# Convert to pandas/numpy
X_train = X_train_pl.to_pandas()
y_train = y_train_pl.to_numpy().ravel()

X_val = X_val_pl.to_pandas()
y_val = y_val_pl.to_numpy().ravel()

X_test = X_test_pl.to_pandas()
y_test = y_test_pl.to_numpy().ravel()

In [ ]:
print(f"\nFeature shapes: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")
print(f"Target shapes: y_train={y_train.shape}, y_val={y_val.shape}, y_test={y_test.shape}")

Beginning of Joel's PCA Elastic Net code

In [ ]:
# Select only numeric columns from X_train and X_test
# X_train = X_train.select(pl.col(pl.NUMERIC_DTYPES))
# X_val = X_val.select(pl.col(pl.NUMERIC_DTYPES))
# X_test = X_test.select(pl.col(pl.NUMERIC_DTYPES))

# print("Scaling Data")

# Initialize the StandardScaler
scaler = StandardScaler()
# Fit the scaler to the training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling Completed")

In [ ]:
# X_train_scaled is scaled training data.
pca_full = PCA(n_components=None, random_state=2025)
pca_full.fit(X_train_scaled)

# Get cumulative explained variance for all components
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

for i in range(1, 9):
    variance = cumulative_variance[i * 100 - 1]
    info_loss = 1 - variance
    print(f"Variance retained with {i * 100} components: {variance:.6%}")
    print(f"Information lost with {i * 100} components: {info_loss:.6%}")

In [ ]:
# First ensure that data is in NumPy format for scikitlearn
def ensure_numpy(data):
    if isinstance(data, pd.DataFrame):
        return data.values
    elif hasattr(data, "to_numpy"):
        return data.to_numpy()
    return data

X_train_scaled = ensure_numpy(X_train_scaled)
y_train = ensure_numpy(y_train)
X_val_scaled = ensure_numpy(X_val_scaled)
y_val = ensure_numpy(y_val)

print("Data shapes:")
print("  X_train_scaled:", X_train_scaled.shape)
print("  y_train:", y_train.shape)
print("  X_val_scaled:", X_val_scaled.shape)
print("  y_val:", y_val.shape)

In [ ]:
# (Optional) Subsample Training Data for Grid Search
subsample_size = 20000
if X_train_scaled.shape[0] > subsample_size:
    np.random.seed(2025)
    sample_indices = np.random.choice(X_train_scaled.shape[0], subsample_size, replace=False)
    X_train_sub = X_train_scaled[sample_indices]
    y_train_sub = y_train[sample_indices]
else:
    X_train_sub = X_train_scaled
    y_train_sub = y_train

print("Subsample shapes:")
print("  X_train_sub:", X_train_sub.shape)
print("  y_train_sub:", y_train_sub.shape)

In [ ]:
# Build a Pipeline with PCA and ElasticNet

pipeline = Pipeline([
    ('pca', PCA()), # ('pca', PCA(n_components=600, random_state=2025))
    ('elasticnet', ElasticNet(random_state=2025, warm_start=True))
])

# Set up the hyperparameter grid for both PCA and ElasticNet.
param_grid = {
    'elasticnet__alpha': [0.005, 0.01, 0.05, 0.1],
    'elasticnet__l1_ratio': [0.0005, 0.001, 0.005, 0.01,],
    'elasticnet__max_iter': [5000, 10000, 20000],
}

cv_object = KFold(n_splits=3, shuffle=True, random_state=2025)

# Using both r2 and negative MSE for scoring, and refitting based on r2.
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_object,
    scoring=['r2', 'neg_mean_squared_error'],
    n_jobs=-1,
    refit='r2',
    verbose=2
)

print("\nStarting GridSearchCV on subsampled training data with PCA...")
grid_search.fit(X_train_sub, y_train_sub)
print("Best parameters found:")
print(grid_search.best_params_)

Building Elastic Net Model

In [ ]:
# New code Joel updated
# Making pca df for train and test
pca = PCA(n_components=600, random_state=2025)
# Fit PCA on training data
X_train_pca = pca.fit_transform(X_train_scaled)
# Apply the same transformation to the test data
X_test_pca = pca.transform(X_test_scaled)

print("Shape after PCA - Training:", X_train_pca.shape)
print("Shape after PCA - Test:", X_test_pca.shape)

# Refit the Final Model on the Full Training Data
# Use the best parameters from the grid search.
elastic_model = ElasticNet(
    alpha=0.05,
    l1_ratio=0.001,
    max_iter=5000,
    random_state=2025,
    warm_start=True
)

print("\nRefitting final model on full training data...")
elastic_model.fit(X_train_pca, y_train)

# Getting evaluation on training set
train_r2 = elastic_model.score(X_train_pca, y_train)
print(f"Training R² on PCA‐space: {train_r2:.3f}")


In [ ]:
# Joel updated
# Predict on Test Set and Create Scatterplot
predictions = elastic_model.predict(X_test_pca)

plt.figure(figsize=(8,6))
plt.scatter(y_test, predictions, alpha=0.5, label="Predictions")
# Convert y_test to a Series using to_series() before using min/max.
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', label="Ideal")
plt.xlabel("Actual Response")
plt.ylabel("Predicted Response")
plt.title("Actual vs Predicted Response on Test Set")
plt.legend()
plt.show()

In [ ]:
# MSE, RMSE, R², and Bootstrapped Confidence Intervals

mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
print("Test Set Metrics:")
print(f"Mean Squared Error (MSE): {mse:.6f}")
print(f"R² Score: {r2:.3f}")

def bootstrap_metric(metric_func, y_true, y_pred, n_iterations=1000, random_state=2025):
    np.random.seed(random_state)
    metric_vals = []
    n = len(y_true)
    for _ in range(n_iterations):
        indices = np.random.randint(0, n, n)
        metric_val = metric_func(y_true[indices], y_pred[indices])
        metric_vals.append(metric_val)
    lower = np.percentile(metric_vals, 2.5)
    upper = np.percentile(metric_vals, 97.5)
    return lower, upper

mse_ci = bootstrap_metric(mean_squared_error, np.array(y_test), np.array(predictions))
r2_ci = bootstrap_metric(r2_score, np.array(y_test), np.array(predictions))

print("\nBootstrapped 95% Confidence Intervals:")
print(f"MSE: {mse_ci[0]:.6f} to {mse_ci[1]:.6f}")
print(f"R²: {r2_ci[0]:.3f} to {r2_ci[1]:.3f}")

In [ ]:
# Original feature names from Polars X_train DataFrame
original_feature_names = X_train.columns

# Build the explainer on the already‑fitted ElasticNet in PC‑space
masker    = shap.maskers.Independent(X_train_pca)
explainer = shap.LinearExplainer(elastic_model, masker)

# Compute SHAP values for test set (in PC‐space)
shap_pca = explainer.shap_values(X_test_pca)

# Bar summary of which PRINCIPAL COMPONENTS matter most
shap.summary_plot(
    shap_pca,
    X_test_pca,
    feature_names=[f"PC_{i+1}" for i in range(X_test_pca.shape[1])],
    plot_type="dot"
)

# Project those PC‑SHAP values back to ORIGINAL FEATURES:
shap_orig = shap_pca.dot(pca.components_)

# Bar summary of which ORIGINAL FEATURES matter most
shap.summary_plot(
    shap_orig,
    X_test_scaled,
    feature_names=original_feature_names,
    plot_type="dot"
)

Beginning of Meghan's Elastic Net code

In [ ]:
# Already dropped above
# X_train = X_train.drop("CODE")
# X_test = X_test.drop("CODE")
# X_val =X_val.drop("CODE")

In [ ]:
tracking_seed = 2025

In [ ]:
l1_candidates = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]
alpha_candidates = np.logspace(-3, 1, 100)

EN_CV = ElasticNetCV(alphas=alpha_candidates,
                     l1_ratio=l1_candidates,
                     cv=5,
                     n_jobs=2,
                     random_state=tracking_seed,
                     max_iter=20000,
                     tol=1e-3,
                     verbose=2)

EN_CV_pipe = Pipeline(
    steps=[('scaler', StandardScaler()),
           ('EN_CV', EN_CV)]
)

EN_CV_pipe.fit(X_train, y_train)

In [ ]:
# Access the fitted ElasticNetCV step
fitted_en_cv = EN_CV_pipe.named_steps['EN_CV']
best_alpha = fitted_en_cv.alpha_
best_l1_ratio = fitted_en_cv.l1_ratio_
coefs = fitted_en_cv.coef_
intercept = fitted_en_cv.intercept_

In [ ]:
# Get the best alpha (as you already have)
best_alpha = 0.0027825594022071257

# Get the best l1_ratio
best_l1_ratio = 0.1

In [ ]:
elasNet_opt = ElasticNet(alpha = best_alpha,
                         l1_ratio = best_l1_ratio,
                         max_iter = 20000,
                         random_state = tracking_seed)

elasNet_pipe_opt = Pipeline([('scaler', StandardScaler()),
                             ('elasNet_opt', elasNet_opt)])

elasNet_pipe_opt.fit(X_train, y_train)

In [ ]:
# # Define the ElasticNet Regression
# elastic_Net = ElasticNet(random_state=2025)

# # Define cross-valdiation object using 3-fold CV
# cv_object = KFold(n_splits = 3,
#                   shuffle = True,
#                   random_state=2025)

# # Define the parameter grid for alpha (learning rate) and l1 ratio
# param_grid_elas = dict({'alpha': [0.008, 0.01, 0.015, 0.1],
#                         'l1_ratio': [0.003, 0.008, 0.01, 0.05],
#                         'max_iter': [8000, 10000, 12000]})

# # Define the Grid Search object that will help us determine the best hyperparameters
# # for our model
# grid_elas = GridSearchCV(elastic_Net,
#                           param_grid_elas,
#                           cv = cv_object,
#                           scoring = ['r2', 'neg_mean_squared_error'],
#                           refit = 'r2',
#                           n_jobs = -1,
#                           verbose = 2)

# ElasNet_pipe = Pipeline(
#     steps=[('scaler', StandardScaler()),
#            ('elastic_net', grid_elas)]
# )

# ElasNet_pipe.fit(X_train, Y_train)
# ElasNet_pipe.named_steps['elastic_net'].best_params_

Create an Elastic Net model with the optimal parameters as determined by the GridSearchCV and train it.

In [ ]:
# opt_alpha = 0.015
# opt_l1_ratio = 0.01
# opt_max_iter = 10000

# elasNet_opt = ElasticNet(alpha = opt_alpha,
#                          l1_ratio = opt_l1_ratio,
#                          max_iter = opt_max_iter,
#                          random_state = 2025)
# elasNet_pipe_opt = Pipeline([('scaler', StandardScaler()),
#                              ('elasNet_opt', elasNet_opt)])

# elasNet_pipe_opt.fit(X_train, y_train)

Use the optimized elastic net model to make predictions on Y_val to see how our model is looking with our new parameters.

In [ ]:
Y_val_pred = elasNet_pipe_opt.predict(X_val)
mse_val = mean_squared_error(y_val, Y_val_pred)
r2_val = r2_score(y_val, Y_val_pred)

print(f"The mean squared error on the validation set is {mse_val}")
print(f"The R^2 score on the validation set is {r2_val}")

Repeat above process on the test set.

In [ ]:
Y_test_pred = elasNet_pipe_opt.predict(X_test)
mse_test = mean_squared_error(y_test, Y_test_pred)
r2_test = r2_score(y_test, Y_test_pred)

print(f"The mean squared error on the test set is {mse_test}")
print(f"The R^2 score on the test set is {r2_test}")

Plot the actual vs predicted values for X_test.

In [ ]:
plt.scatter(y_test, Y_test_pred, alpha = 0.5)
plt.xlabel("Actual Proportion of Income")
plt.ylabel("Predicted Proportion of Income")
min_val = min(y_test.min(), Y_test_pred.min())
max_val = max(y_test.max(), Y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal (y=x)')
plt.show()

In [ ]:
# Calculating residuals (validation data)
y_test_pred = elasNet_pipe_opt.predict(X_test)
residuals = y_test - y_test_pred

In [ ]:
# Residual plot (validation data)

plt.figure(figsize=(8, 5))
plt.scatter(y_test_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.grid(True)
plt.show()

In [ ]:
# Make predictions on the test set using the best model
y_test_pred = elasNet_pipe_opt.predict(X_test)

# Evaluate the model on the test set
mse = mean_squared_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print(f"  Mean Squared Error (MSE): {mse:.4f}")
print(f"  R-squared (R2 Score):   {r2:.4f}")

# Bootstrap confidence intervals for MSE and R^2

# Number of bootstrap iterations
n_bootstrap = 1000

# Lists to store the bootstrapped values
mse_bootstrap = []
r2_bootstrap = []

# For reproducibility
np.random.seed(2025)

# Do bootstrap resampling on the test set
for i in range(n_bootstrap):
    # Resampling w/ repl
    sample_indices = np.random.choice(len(y_test), size=len(y_test), replace=True)

    # Bootstrap samples for y_test and the predictions
    y_test_boot = y_test[sample_indices]
    y_pred_boot = y_test_pred[sample_indices]

    # Compute and store the performance scores for the current bootstrap sample
    mse_bootstrap.append(mean_squared_error(y_test_boot, y_pred_boot))
    r2_bootstrap.append(r2_score(y_test_boot, y_pred_boot))

# Convert the results to numpy arrays
mse_bootstrap = np.array(mse_bootstrap)
r2_bootstrap = np.array(r2_bootstrap)

# Calculate the 2.5th and 97.5th percentiles for the bootstrap distributions
mse_ci_lower, mse_ci_upper = np.percentile(mse_bootstrap, [2.5, 97.5])
r2_ci_lower, r2_ci_upper = np.percentile(r2_bootstrap, [2.5, 97.5])

# Report the bootstrap confidence intervals
print("\nBootstrap 95% Confidence Intervals:")
print(f"Mean Squared Error (MSE): [{mse_ci_lower:.4f}, {mse_ci_upper:.4f}]")
print(f"R-squared (R2 Score): [{r2_ci_lower:.4f}, {r2_ci_upper:.4f}]")

Extract Elastic Net coefficients

In [ ]:
# Extract coefficients and intercept
coefficients = elasNet_pipe_opt.named_steps["elasNet_opt"].coef_
intercept = elasNet_pipe_opt.named_steps["elasNet_opt"].intercept_

# Create Series with feature names
coef_series = pd.Series(coefficients, index=X_train_pl.to_pandas().columns)

# Filter non-zero coefficients
non_zero_coefs = coef_series[coef_series != 0]

# Sort by absolute value (strongest effects first)
sorted_coefs_abs = non_zero_coefs.reindex(non_zero_coefs.abs().sort_values(ascending=False).index)

# print("Intercept:", intercept)
# print("Non-zero Coefficients:\n", non_zero_coefs)

Looking at the most important variables

In [ ]:
(
    sorted_coefs_abs.reset_index().rename(columns = {"index" : "Variable"})
    .merge(metadata_df, on = "Variable", how = "left")
    .head(20)
)

## Define XGBoost Pipeline and Grid Search

Initialize the XGBoost regressor

In [ ]:
XGB_regressor = XGBRegressor(
    random_state=2025 # Seed for reproducibility
)

Set up a basic pipeline for XGBoost

In [ ]:
# Define a basic pipeline; just contains the regressor right now
xgb_pipeline = Pipeline(steps=[('regressor', XGB_regressor)])

Set up the hyperparameter search grid

In [ ]:
param_grid_xgb = {
    'regressor__n_estimators': [50, 100, 200, 300, 400],
    'regressor__learning_rate': [0.025, 0.05, 0.1, 0.15, 0.2],
    'regressor__max_depth': [1, 3, 5, 7, 9],
    # 'regressor__objective' : ['reg:squarederror', 'r2'],
    # 'regressor__subsample': [0.7, 0.9, 1.0],
    # 'regressor__colsample_bytree': [0.7, 0.9, 1.0],
    # 'regressor__gamma': [0, 1],
    # 'regressor__reg_alpha': [0, 1],
    # 'regressor__reg_lambda': [0, 1]
}

Set up the cross-validation parameters

In [ ]:
# Define cross-validation scheme
cv_object_xgb = KFold(n_splits = 5, shuffle = True, random_state = 2025)

Initialize the grid search object

In [ ]:
# Define GridSearchCV object
grid_search_xgb = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid_xgb,
    scoring='neg_mean_squared_error', # Or 'r2'
    cv=cv_object_xgb,
    n_jobs=-1,          # Use all cores for CV folds
    verbose=3,          # Show progress
    refit=True          # Refit the best model on the whole training set automatically
)

Run the grid search

In [ ]:
# Run GridSearchCV
grid_search_xgb.fit(X_train, y_train)

## Save Model Results
- Export GridSearchCV object for future use.

In [ ]:
import pickle

# with open("D:\\Downloads\\2025-04-19_gridsearchcv_results_0009.pkl", 'wb') as f:
#     pickle.dump(grid_search_xgb, f)

with open("/content/drive/My Drive/DS3000/coursework_data/2025-04-20_gridsearchcv_results_0011.pkl", 'wb') as f:
    pickle.dump(grid_search_xgb, f)

Load the GridSearchCV result from a pickle file instead of re-running it

In [ ]:
# import pickle

# # Open the pickle file in binary read mode
# with open("D:\\Downloads\\2025-04-19_gridsearchcv_results_0009.pkl", 'rb') as file:
#     grid_search_xgb = pickle.load(file)

Save the best hyperparameters that were found

In [ ]:
# Save the best prarameters
best_params = grid_search_xgb.best_params_

Show the best hyperparameters found

In [ ]:
# Show best params found by GridSearchCV
print("\nBest parameters found by GridSearchCV:")
print(best_params)

## Train Final Model

In [ ]:
# Initialize a new XGBoost regressor using the best hyperparameters we found
final_xgb_model = XGBRegressor(
    n_estimators=best_params['regressor__n_estimators'],
    learning_rate=best_params['regressor__learning_rate'],
    max_depth=best_params['regressor__max_depth'],
    random_state=2025,
    n_jobs=2,
    verbosity=2
)

# Train the final model
final_xgb_model.fit(X_train, y_train)

Calculate residuals on the test set

In [ ]:
y_test_pred = final_xgb_model.predict(X_test)
residuals = y_test - y_test_pred

Create a residual plot

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.grid(True)
plt.show()

## Evaluate Final Model on Test Set

In [ ]:
# Make predictions on the test set using the best model
y_pred_test = final_xgb_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# Evaluate the model using regression metrics
mse = mean_squared_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

print(f"\nModel Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse:.4f}")
print(f"  R-squared (R2 Score):   {r2:.4f}")

## Visualize Predictions

In [ ]:
# Scatter plot of Actual vs Predicted values
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_test, alpha=0.5, edgecolor='k', s=50)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], '--r', linewidth=2, label='Theoretical Perfect Fit (y=x)') # Diagonal line for reference
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs. Predicted Target (XGBoost)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## Confidence Intervals via Bootstrapping
- Generate 95% confidence intervals for MSE and R².

In [ ]:
# Make predictions on the test set using the best model
y_pred_test = final_xgb_model.predict(X_test)

# Evaluate the model on the test set
mse = mean_squared_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

print(f"  Mean Squared Error (MSE): {mse:.4f}")
print(f"  R-squared (R2 Score):   {r2:.4f}")

# Bootstrap confidence intervals for MSE and R^2

# Number of bootstrap iterations
n_bootstrap = 1000

# Lists to store the bootstrapped values
mse_bootstrap = []
r2_bootstrap = []

# For reproducibility
np.random.seed(2025)

# Do bootstrap resampling on the test set
for i in range(n_bootstrap):
    # Resampling w/ repl
    sample_indices = np.random.choice(len(y_test), size=len(y_test), replace=True)

    # Bootstrap samples for y_test and the predictions
    y_test_boot = y_test[sample_indices]
    y_pred_boot = y_pred_test[sample_indices]

    # Compute and store the performance scores for the current bootstrap sample
    mse_bootstrap.append(mean_squared_error(y_test_boot, y_pred_boot))
    r2_bootstrap.append(r2_score(y_test_boot, y_pred_boot))

# Convert the results to numpy arrays
mse_bootstrap = np.array(mse_bootstrap)
r2_bootstrap = np.array(r2_bootstrap)

# Calculate the 2.5th and 97.5th percentiles for the bootstrap distributions
mse_ci_lower, mse_ci_upper = np.percentile(mse_bootstrap, [2.5, 97.5])
r2_ci_lower, r2_ci_upper = np.percentile(r2_bootstrap, [2.5, 97.5])

# Report the bootstrap confidence intervals
print("\nBootstrap 95% Confidence Intervals:")
print(f"Mean Squared Error (MSE): [{mse_ci_lower:.4f}, {mse_ci_upper:.4f}]")
print(f"R-squared (R2 Score): [{r2_ci_lower:.4f}, {r2_ci_upper:.4f}]")


# Model Interpretability with SHAP

## Generate SHAP Values

In [ ]:
# Create a SHAP explainer for the XGBoost model
explainer = shap.TreeExplainer(final_xgb_model)

# Compute SHAP values for the test set
shap_values = explainer.shap_values(X_test)

## SHAP Summary Plot

In [ ]:
# Display a summary plot of feature importance
shap.summary_plot(shap_values, X_test)

## Enhanced SHAP Summary with Feature Descriptions

In [ ]:
# Make sure columns are named correctly
metadata_df.columns = ['Variable', 'Description']

# Create a dict for mapping variable names to their descriptions
var_to_desc = dict(zip(metadata_df['Variable'], metadata_df['Description']))

# Rename columns in X_test using the dictionary
# If a variable isn't in metadata, just use the original name
X_test_renamed = X_test.rename(columns=lambda col: var_to_desc.get(col, col))

# print(X_test_renamed.columns.tolist())

shap.summary_plot(shap_values, X_test_renamed, max_display=30)

## Rank Features by SHAP Contribution

In [ ]:
# # Calculate mean absolute SHAP values per feature
# mean_abs_shap = np.abs(shap_values).mean(axis=0)

# shap_importance_df = pd.DataFrame({
#     'Description': X_test_renamed.columns,
#     'Mean_Absolute_SHAP': mean_abs_shap
# }).sort_values(by='Mean_Absolute_SHAP', ascending=False)

# top_n = 30
# shap_top = shap_importance_df.head(top_n)
# print(shap_top)

# Mean SHAP values (not absolute)
mean_shap = shap_values.mean(axis=0)

# Mean absolute SHAP values
mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Create a DataFrame to hold the results
shap_importance_df = pd.DataFrame({
    'Description': X_test_renamed.columns,
    'Mean_SHAP': mean_shap,
    'Mean_Absolute_SHAP': mean_abs_shap
})

# Sort for top absolute importance (regardless of direction)
top_30_abs = shap_importance_df.sort_values(by='Mean_Absolute_SHAP', ascending=False).head(30)

# Sort for top positive contribution
top_20_pos = shap_importance_df.sort_values(by='Mean_SHAP', ascending=False).head(20)

# Sort for top negative contribution
top_20_neg = shap_importance_df.sort_values(by='Mean_SHAP', ascending=True).head(20)

# Print results
print("\nTop 30 Most Important Features (by |mean SHAP|):")
print(top_30_abs)

print("\nTop 20 Positive SHAP Contributors:")
print(top_20_pos)

print("\nTop 20 Negative SHAP Contributors:")
print(top_20_neg)


## Dependence Plot for a chosen feature

In [ ]:
# Show a dependence plot for top feature
feature_name = 'ECYMTNAVG'
shap.dependence_plot(feature_name, shap_values, X_test)

In [ ]:
# Show a dependence plot for 2nd top feature
feature_name = 'ECYHNI200P'
shap.dependence_plot(feature_name, shap_values, X_test)

In [ ]:
# Show a dependence plot for 3rd top feature
feature_name = 'ECYINDPUBL'
shap.dependence_plot(feature_name, shap_values, X_test)

In [ ]:
# Show a dependence plot for 4th top feature
feature_name = 'ECYMTN6574'
shap.dependence_plot(feature_name, shap_values, X_test)

In [ ]:
# Show a dependence plot for 5th top feature
feature_name = 'ECYMTN7584'
shap.dependence_plot(feature_name, shap_values, X_test)